In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("test").getOrCreate()
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# Customers DataFrame
customer_data = [(1, "Alice"), (2, "Bob"), (3, "Charlie"), (4, "David")]
customers = spark.createDataFrame(customer_data, ["cust_id", "name"])

# Orders DataFrame (skewed: many orders for cust_id=1)
order_data = [(101, 1), (102, 1), (103, 1), (104, 2), (105, 3)]
orders = spark.createDataFrame(order_data, ["order_id", "cust_id"])
display(customers)
display(orders)


cust_id,name
1,Alice
2,Bob
3,Charlie
4,David


order_id,cust_id
101,1
102,1
103,1
104,2
105,3


In [0]:
from pyspark.sql.functions import col, expr

# Add a random salt key to orders
orders_salted = orders.withColumn("salt", expr("floor(rand() * 5)"))
display(orders_salted)

order_id,cust_id,salt
101,1,1
102,1,3
103,1,3
104,2,1
105,3,1


In [0]:
# Duplicate customers with all salt values
salt_range = spark.range(0, 5).withColumnRenamed("id", "salt")
display(salt_range)
customers_salted = customers.crossJoin(salt_range)
display(customers_salted)


salt
0
1
2
3
4


cust_id,name,salt
1,Alice,0
1,Alice,1
1,Alice,2
1,Alice,3
1,Alice,4
2,Bob,0
2,Bob,1
2,Bob,2
2,Bob,3
2,Bob,4


In [0]:
# Join on both cust_id and salt
joined_df = customers_salted.join(
    orders_salted,
    (customers_salted.cust_id == orders_salted.cust_id) &
    (customers_salted.salt == orders_salted.salt),
    "inner"
)

joined_df.show()


+-------+-------+----+--------+-------+----+
|cust_id|   name|salt|order_id|cust_id|salt|
+-------+-------+----+--------+-------+----+
|      1|  Alice|   1|     101|      1|   1|
|      1|  Alice|   3|     102|      1|   3|
|      1|  Alice|   3|     103|      1|   3|
|      2|    Bob|   1|     104|      2|   1|
|      3|Charlie|   1|     105|      3|   1|
+-------+-------+----+--------+-------+----+

